# 04 · Telemetry: where is the difference made?

**Mission covered:** 7

**Question:** on their fastest qualifying lap, **where** did Leclerc gain the advantage over Piastri, and **how**?

**Telemetry** is the data from the car's sensors, recorded several times per second:
`Speed` (km/h), `Throttle` (%), `Brake` (on/off), `nGear` (gear), `RPM` (engine speed).

**Done when** you can point to a part of the track and say: "this is where X gains time, because…".

**How to use this notebook:** each step starts with a ✏️ **Exercise**. Try to answer it yourself
(add a new cell below it) before reading the ✅ **Solution** and the 💡 **Explanation** that follow.

## Setup

These cells are the same as in `01_exploration.ipynb`: imports, FastF1 cache and the `show` function.
The exercises and explanations about them are in notebook 01.

> 🔁 Copying this block into every notebook is not ideal: if it is fixed in one place, it has to be fixed everywhere.
> This is exactly the problem that **mission 8** will solve (moving this code into `src/`).

In [ ]:
import os

import fastf1
import fastf1.plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# The relative path "../cache" is only correct if the notebook runs from notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Unexpected working directory: {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("FastF1 cache:", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

In [ ]:
def show(df, columns=None, n=None):
    '''Display a DataFrame as a Jupyter table, without truncated columns.

    columns : list of column names to keep (all by default)
    n       : number of rows to display (all by default)
    '''
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

### Loading the session


We use **qualifying** (`"Q"`): each driver does a flat-out lap,
which makes a telemetry comparison meaningful.

In [ ]:
YEAR = 2024
GRAND_PRIX = "Monaco"
SESSION_TYPE = "Q"

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load()
print(session.event["EventName"], "-", session.name)

In [ ]:
# Without a Formula 1 account sign-in, FastF1 only retrieves the results:
# accessing session.laps then raises a DataNotLoadedError.
try:
    laps_count = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    laps_count = 0

if laps_count == 0:
    print("⚠️ No laps loaded. Sign in from a terminal with:")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {laps_count} laps loaded")

## 7.1 · Each driver's fastest lap

> ✏️ **Exercise**
>
> Choose two drivers and get the **fastest lap** of each one.
> Hint: the `pick_drivers` and `pick_fastest` methods of `Laps`.
>
> To avoid duplicating code, store both laps in a **dictionary** `{driver: lap}`.

In [ ]:
# ✅ Solution
DRIVER_1 = "LEC"
DRIVER_2 = "PIA"
DRIVERS = [DRIVER_1, DRIVER_2]

fastest_laps = {driver: session.laps.pick_drivers(driver).pick_fastest() for driver in DRIVERS}

for driver, lap in fastest_laps.items():
    print(f"{driver}: {lap['LapTime'].total_seconds():.3f} s (lap {int(lap['LapNumber'])})")

💡 **Explanation**

`pick_fastest` returns a `Lap` object, i.e. a single row. The dictionary comprehension
`{driver: ... for driver in DRIVERS}` builds the dictionary in one line.

## 7.2 · Comparing sectors

> ✏️ **Exercise**
>
> A lap is split into **three officially timed sectors** (`Sector1Time`, `Sector2Time`, `Sector3Time`).
> Build a table with the sector times of both drivers, and a column with the gap between them.
>
> In which sector does each driver gain time?

In [ ]:
# ✅ Solution
sectors = pd.DataFrame(
    {driver: [lap[f"Sector{i}Time"].total_seconds() for i in (1, 2, 3)]
     for driver, lap in fastest_laps.items()},
    index=["Sector 1", "Sector 2", "Sector 3"],
)
sectors[f"Gap {DRIVER_1} - {DRIVER_2} (s)"] = sectors[DRIVER_1] - sectors[DRIVER_2]
show(sectors.round(3))

💡 **Explanation**

Sector times are the most reliable measure to find out where a driver gains time:
Piastri is faster in sector 1, Leclerc gains most of his advantage in sector 2.

## 7.3 · Getting the telemetry

> ✏️ **Exercise**
>
> For each fastest lap, get the car data and add a `Distance` column.
> Hints: `get_car_data()` and `add_distance()`.
>
> Explore the result as in mission 4 (`.shape`, `.columns`, `show(..., n=5)`).
> Do both laps end at the same total distance? Why not?

In [ ]:
# ✅ Solution
telemetry = {driver: lap.get_car_data().add_distance() for driver, lap in fastest_laps.items()}

show(telemetry[DRIVER_1], n=5)
for driver, tel in telemetry.items():
    print(f"{driver}: {len(tel)} samples, computed total distance {tel['Distance'].max():.0f} m")

💡 **Explanation**

- `get_car_data()` returns the sensor measurements during the lap: one row per sample.
- `add_distance()` adds a `Distance` column (in metres from the start of the lap).

⚠️ The distance is **computed** by adding up speed × time between two samples: small errors accumulate.
The two laps therefore do not end at exactly the same total distance (about 50 m apart over 3.3 km).
The curves can drift slightly towards the end of the lap: this is why sector times come first.

## 7.4 · Plotting speed against distance

> ✏️ **Exercise**
>
> Plot the speed of both drivers against the distance, on the same chart.
>
> 1. Why use the **distance** and not the time on the X axis?
> 2. Bonus: mark the corners with vertical lines. Look for `session.get_circuit_info().corners` and `ax.axvline`.

In [ ]:
# ✅ Solution
corners = session.get_circuit_info().corners

fig_speed, ax = plt.subplots(figsize=(14, 5))

for driver, tel in telemetry.items():
    ax.plot(tel["Distance"], tel["Speed"], label=driver,
            color=fastf1.plotting.get_driver_color(driver, session))

for _, corner in corners.iterrows():
    ax.axvline(corner["Distance"], color="grey", linestyle=":", linewidth=0.8)
    ax.text(corner["Distance"], 30, f"{corner['Number']}{corner['Letter']}",
            ha="center", fontsize=8, color="grey")

ax.set_xlabel("Distance (m)")
ax.set_ylabel("Speed (km/h)")
ax.set_title(f"Fastest qualifying lap: {DRIVER_1} vs {DRIVER_2}")
ax.legend()
plt.show()

💡 **Explanation**

1. After 10 seconds, two drivers are not at the same point on the track.
   Comparing "their speed at metre 500" really compares the same corner.
2. The vertical dotted lines and their numbers mark the corners of the circuit.

## 7.5 · Speed, throttle and brake

> ✏️ **Exercise**
>
> Draw three stacked charts sharing the same X axis: speed, throttle and brake.
> Hint: `fig, axes = plt.subplots(3, 1, sharex=True)`.
>
> `Brake` is a boolean: how can you plot it as 0 / 1?

In [ ]:
# ✅ Solution
fig_inputs, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for driver, tel in telemetry.items():
    colour = fastf1.plotting.get_driver_color(driver, session)
    axes[0].plot(tel["Distance"], tel["Speed"], color=colour, label=driver)
    axes[1].plot(tel["Distance"], tel["Throttle"], color=colour)
    axes[2].plot(tel["Distance"], tel["Brake"].astype(int), color=colour)

axes[0].set_ylabel("Speed (km/h)")
axes[1].set_ylabel("Throttle (%)")
axes[2].set_ylabel("Brake (0/1)")
axes[2].set_xlabel("Distance (m)")
axes[0].legend()
fig_inputs.suptitle(f"Fastest lap telemetry: {DRIVER_1} vs {DRIVER_2}")
plt.show()

💡 **Explanation**

With `sharex=True`, you can read vertically what each driver does at the same point on the track.
`.astype(int)` turns `True` / `False` into `1` / `0`.

Where the curves separate, something happens:
- a lower speed point further down the track: the driver **brakes later**;
- a higher top speed: less downforce or a stronger engine;
- full throttle earlier: a better **corner exit**.

## 7.6 · Numerical summary

> ✏️ **Exercise**
>
> For each driver, compute the top speed, the minimum speed, the share of samples at full throttle
> (`Throttle` ≥ 99) and the share of samples on the brakes.
>
> Hint: the mean of a boolean column is the share of `True` values.

In [ ]:
# ✅ Solution
summary = pd.DataFrame({
    driver: {
        "Top speed (km/h)": tel["Speed"].max(),
        "Minimum speed (km/h)": tel["Speed"].min(),
        "Full throttle (% of samples)": round((tel["Throttle"] >= 99).mean() * 100, 1),
        "Braking (% of samples)": round(tel["Brake"].mean() * 100, 1),
    }
    for driver, tel in telemetry.items()
})
show(summary)

💡 **Explanation**

`(tel["Throttle"] >= 99)` is a boolean column: its mean is the share of samples at full throttle.
`Brake` is already a boolean, so its mean directly gives the share of samples on the brakes.

## 7.7 · Exporting

> ✏️ **Exercise**
>
> Save both telemetry charts in `image/`, as in notebook 03.

In [ ]:
# ✅ Solution
image_dir = os.path.abspath("../image")
os.makedirs(image_dir, exist_ok=True)

for name, figure in {"telemetry_speed.png": fig_speed, "telemetry_inputs.png": fig_inputs}.items():
    path = os.path.join(image_dir, name)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    print("Exported:", path)

## Conclusion

> ✏️ **Exercise**
>
> Where does Leclerc make the difference, and how? What are the limits of this analysis?

💡 **Conclusion**

- **Leclerc** takes pole by **0.154 s** over Piastri (1:10.270 vs 1:10.424).
- **Piastri is faster in sector 1** (0.083 s ahead): he is also quicker through the first two
  speed traps (`SpeedI1` and `SpeedI2`).
- **Leclerc makes the difference in sector 2**: he gains **0.206 s** there, most of his advantage.
  Sector 3 is almost identical (0.031 s in Leclerc's favour).
- Over the whole lap, Leclerc spends **more time at full throttle** (about 46 % of samples vs 41 %):
  he gets back on the throttle earlier when exiting corners.
- **Limitation of the analysis**: the computed distance differs by about 50 m between the two laps. To locate a corner precisely,
  sector times are more reliable than overlaying the curves towards the end of the lap.